# Diet Optimization - MIP Implementation

## What's New in This Version
- 🔢 **Mixed-Integer Program**: Binary food selection + continuous servings
- 🎯 **Smarter Decisions**: First choose WHICH foods, then HOW MUCH
- 📊 **Variety Control**: Must select 8-15 different food types
- 🔗 **Logical Linking**: Can't have servings without selection (y=1 → x>0)

## Model Improvements vs. LP
| Feature | LP (Old) | MIP (New) |
|---------|----------|------------|
| Food selection | Implicit (x>0) | Explicit binary variable (y) |
| Variety control | Soft (max servings) | Hard (min/max food types) |
| Realism | Medium | High |
| Solve time | Fast (~0.01s) | Medium (~1-5s) |
| Shadow prices | Yes | No (MIP limitation) |

## Future Extensions
- 📅 **Multi-period**: Weekly meal planning (7-day optimization)
- 🎲 **Stochastic**: Price/satisfaction scenarios
- 📈 **Sensitivity**: Parametric analysis for MIP


# Diet Optimization - Mixed-Integer Linear Program (MIP)

## Model Type: MIP
**Binary decisions** (food selection) + **Continuous decisions** (servings)

## Objective
Minimize total cost subject to nutritional, budget, satisfaction, and variety constraints.

## Decision Variables
- **y[i]**: Binary (0/1) - Is food i selected?
- **x[i]**: Continuous - Servings of food i (only if selected)

## Key Constraints
- **Nutritional**: 17 nutrients with min/max bounds
- **Budget**: $200-$400 total
- **Satisfaction**: Minimum threshold (300)
- **Variety**: 8-15 different food types
- **Linking**: x[i] > 0 only if y[i] = 1
- **Servings**: 0.5-3.0 servings per selected food

## Why MIP?
- ✅ **Realistic**: Can't eat "0.3 types of food" - it's binary
- ✅ **Flexible**: Separate what to eat (binary) from how much (continuous)
- ✅ **Interpretable**: Clear selection decisions
- ⚠️ **Trade-off**: Slower to solve than LP, no shadow prices

In [1]:
# Import libraries and initialize GAMSPy container
import numpy as np
import pandas as pd
import gamspy as gp
import gamspy.math as gpm
import sys

gp.set_options({'USE_PY_VAR_NAME': 'yes'})
m = gp.Container()

In [2]:
# Define model structure: nutrients, restaurants, and menu items
expanded_nutrients = [
    "Calories", "Protein", "Carbs", "Fat", "SaturatedFat", "TransFat", "Sugars",
    "Sodium", "Fiber", "VitaminA", "VitaminC", "VitaminD", "Calcium", "Iron", 
    "Potassium", "Cholesterol", "Caffeine"
]

restaurants_dict = {
    "Chipotle": ["Chicken_Burrito", "Steak_Bowl", "Veggie_Tacos", "Chicken_Salad"],
    "Subway": ["Turkey_Sandwich", "Veggie_Delite", "Chicken_Teriyaki", "Meatball_Marinara"],
    "McDonalds": ["Big_Mac", "Quarter_Pounder", "Chicken_Nuggets", "French_Fries"],
    "PizzaHut": ["Pepperoni_Pizza", "Cheese_Pizza", "Veggie_Pizza", "Breadsticks"],
    "TacoBell": ["Crunchwrap", "Taco", "Burrito", "Nachos"],
    "Starbucks": ["Latte", "Cappuccino", "Frappuccino", "Muffin", "Croissant"],
    "Dunkin": ["Coffee", "Donut", "Bagel"],
    "DailyScoop": ["Vanilla_Cone", "Chocolate_Sundae", "Strawberry_Scoop", "Cookie_Dough"],
    "ColdStone": ["IceCream_Cake", "Smoothie", "Milkshake"]
}

restaurants_list = list(restaurants_dict.keys())
menu_items_data = [(r, item) for r, items in restaurants_dict.items() for item in items]
food_list = [f"{r}_{item}" for r, items in restaurants_dict.items() for item in items]


In [3]:
# Load nutritional data from CSV
df = pd.read_csv("nutrient_data.csv")

# Create nutrient dictionary for each food
nutrient_values = {}
for _, row in df.iterrows():
    item = row['Restaurant_MenuItem']
    nutrient_values[item] = {
        "Calories": row['Calories'], "Protein": row['Protein'], "Carbs": row['Carbs'], "Fat": row['Fat'],
        "SaturatedFat": row['SaturatedFat'], "TransFat": row['TransFat'], "Sugars": row['Sugars'],
        "Sodium": row['Sodium'], "Fiber": row['Fiber'], "VitaminA": row['VitaminA'], "VitaminC": row['VitaminC'],
        "VitaminD": row['VitaminD'], "Calcium": row['Calcium'], "Iron": row['Iron'], "Potassium": row['Potassium'],
        "Cholesterol": row['Cholesterol'], "Caffeine": row['Caffeine']
    }

# Expand to (food, nutrient, value) tuples for GAMSPy
nutrient_data_expanded = [(food, nutrient, nutrient_values[food].get(nutrient, 0)) 
                          for food in food_list for nutrient in expanded_nutrients]


In [4]:
# FEASIBILITY DIAGNOSTIC: Check if nutrient constraints are achievable
# Note: Run this after loading all parameters to diagnose infeasibility

# This cell is for diagnostic purposes only
# Uncomment and run if model is infeasible to identify issues

# nutrient_df_check = pd.read_csv("nutrient_data.csv")
# constraints_df_check = pd.read_csv("nutrient_constraints.csv")
# config_df_check = pd.read_csv("model_config.csv")
# max_servings = float(config_df_check[config_df_check['Parameter']=='max_servings_per_food']['Value'].values[0])
# max_foods = int(config_df_check[config_df_check['Parameter']=='max_food_types']['Value'].values[0])

# print(f"Checking feasibility (max {max_foods} foods at {max_servings} servings each)...")
# for _, constraint_row in constraints_df_check.iterrows():
#     nutrient_name = constraint_row['Nutrient']
#     nmin = constraint_row['Nmin']
#     top_foods = nutrient_df_check.nlargest(max_foods, nutrient_name)[nutrient_name]
#     max_possible = top_foods.sum() * max_servings
#     if max_possible < nmin:
#         print(f"⚠️ {nutrient_name}: Need {nmin}, max possible {max_possible:.1f}")

print("Feasibility diagnostic ready (uncomment to run if model is infeasible)")

Feasibility diagnostic ready (uncomment to run if model is infeasible)


In [5]:
#from IPython.display import HTML, display
#display(HTML('<iframe src="https://www.nutritionix.com/subway/nutrition-calculator" width="100%" height="800"></iframe>'))


In [6]:
# Create GAMSPy sets
restaurants = gp.Set(m, name="restaurants", records=restaurants_list)
foods = gp.Set(m, name="foods", records=food_list)
nutrients = gp.Set(m, name="nutrients", records=expanded_nutrients)

# Load food prices from CSV
prices_df = pd.read_csv("food_prices.csv", dtype={'Food': str, 'Restaurant': str})
price_per_serving_data = prices_df[['Food', 'Price']].copy()
price_per_serving_data.columns = ['foods', 'value']
price_per_serving = gp.Parameter(m, name="price_per_serving", domain=[foods], records=price_per_serving_data)

# Load nutritional content per serving
nutrient_per_serving = gp.Parameter(m, name="nutrient_per_serving", domain=[foods, nutrients], records=nutrient_data_expanded)

# Load model configuration parameters
config_df = pd.read_csv("model_config.csv")
config_values = config_df.set_index('Parameter')['Value'].to_dict()

print(f"✅ Loaded: {len(food_list)} foods, {len(expanded_nutrients)} nutrients")


✅ Loaded: 35 foods, 17 nutrients


In [7]:
# Load satisfaction parameters from CSV
satisfaction_df = pd.read_csv("food_satisfaction.csv", dtype={'Food': str})

base_satisfaction_data = satisfaction_df[['Food', 'Base_Satisfaction']].copy()
base_satisfaction_data.columns = ['foods', 'value']
base_satisfaction = gp.Parameter(m, name="base_satisfaction", domain=[foods], records=base_satisfaction_data)

satisfaction_penalty_data = satisfaction_df[['Food', 'Habituation_Rate']].copy()
satisfaction_penalty_data.columns = ['foods', 'value']
satisfaction_penalty = gp.Parameter(m, name="satisfaction_penalty", domain=[foods], records=satisfaction_penalty_data)

min_satisfaction = gp.Parameter(m, name="min_satisfaction", records=float(config_values['min_satisfaction']))

In [8]:
# Load nutrient constraints (min/max bounds) from CSV
constraints_df = pd.read_csv("nutrient_constraints.csv", dtype={'Nutrient': str})

Nmin_data = constraints_df[['Nutrient', 'Nmin']].copy()
Nmin_data.columns = ['nutrients', 'value']
Nmin = gp.Parameter(m, name="Nmin", domain=[nutrients], records=Nmin_data)

Nmax_data = constraints_df[['Nutrient', 'Nmax']].copy()
Nmax_data.columns = ['nutrients', 'value']
Nmax = gp.Parameter(m, name="Nmax", domain=[nutrients], records=Nmax_data)


In [9]:
# Extract model parameters from config
w_cost = gp.Parameter(m, name="w_cost", records=float(config_values['w_cost']))
w_satisfaction = gp.Parameter(m, name="w_satisfaction", records=float(config_values['w_satisfaction']))
max_servings_per_food = gp.Parameter(m, name="max_servings_per_food", records=float(config_values['max_servings_per_food']))

# Budget constraints
budget_min = float(config_values['budget_min'])
budget_max = float(config_values['budget_max'])

# MIP-specific parameters
min_food_types = int(config_values['min_food_types'])
max_food_types = int(config_values['max_food_types'])
min_servings_if_selected = float(config_values['min_servings_if_selected'])
variety_bonus_weight = float(config_values['variety_bonus_weight'])

In [10]:
# Load food serving bounds from CSV file
# CSV Format: Food,Fmin,Fmax,Description
bounds_csv = "food_bounds.csv"
bounds_df = pd.read_csv(bounds_csv, dtype={'Food': str})

# Create DataFrames for Fmin and Fmax (column names must match domain set name)
Fmin_data = bounds_df[['Food', 'Fmin']].copy()
Fmin_data.columns = ['foods', 'value']  # Column name 'foods' matches the set name

Fmax_data = bounds_df[['Food', 'Fmax']].copy()
Fmax_data.columns = ['foods', 'value']  # Column name 'foods' matches the set name

# Fmini = minimum number of required servings of food i, ∀i∈F
Fmin = gp.Parameter(m, name="Fmin", domain=[foods], records=Fmin_data)

Fmax = gp.Parameter(m, name="Fmax", domain=[foods], records=Fmax_data)

In [11]:
# Decision Variables (Mixed-Integer Program)

# Binary variable: 1 if food is selected, 0 otherwise
y = gp.Variable(m, name="y", domain=[foods], type="binary",
                description="1 if food i is selected")

# Continuous variable: servings of each selected food
x = gp.Variable(m, name="x", domain=[foods], type="positive",
                description="servings of food i")
x.lo[foods] = 0
x.up[foods] = max_servings_per_food  # Max servings per food

In [12]:
# Equations (Constraints)

# Constraint Set 1: For each nutrient j∈N, at least meet the minimum required level
# ∑(i∈F) aij*xi ≥ Nminj, ∀j∈N
nutrient_min = gp.Equation(m, name="nutrient_min", domain=[nutrients], description="minimum nutrient requirements")
nutrient_min[nutrients] = gp.Sum(foods, nutrient_per_serving[foods, nutrients] * x[foods]) >= Nmin[nutrients]

# Constraint Set 2: For each nutrient j∈N, do not exceed the maximum allowable level
# ∑(i∈F) aij*xi ≤ Nmaxj, ∀j∈N
nutrient_max = gp.Equation(m, name="nutrient_max", domain=[nutrients], description="maximum nutrient limits")
nutrient_max[nutrients] = gp.Sum(foods, nutrient_per_serving[foods, nutrients] * x[foods]) <= Nmax[nutrients]

# Constraint: Total cost (budget) must be within configured limits
cost_min = gp.Equation(m, name="cost_min", description=f"minimum budget constraint (${budget_min})")
cost_min[:] = gp.Sum(foods, price_per_serving[foods] * x[foods]) >= budget_min

cost_max = gp.Equation(m, name="cost_max", description=f"maximum budget constraint (${budget_max})")
cost_max[:] = gp.Sum(foods, price_per_serving[foods] * x[foods]) <= budget_max
# Note: Constraint Set 3 (xi ≥ Fmini) and Constraint Set 4 (xi ≤ Fmaxi) 
# are already handled by the variable bounds set in Cell 4

# MIP Linking Constraints: Connect binary selection (y) to continuous servings (x)

# Upper bound linking: If y=0, then x must be 0; If y=1, x can be up to max_servings
link_upper = gp.Equation(m, name="link_upper", domain=[foods],
                        description="Link selection to servings")
link_upper[foods] = x[foods] <= max_servings_per_food * y[foods]

# Lower bound linking (OPTIONAL - can cause infeasibility)
# Uncomment to enforce: if food selected, must have at least min servings
# link_lower = gp.Equation(m, name="link_lower", domain=[foods])
# link_lower[foods] = x[foods] >= min_servings_if_selected * y[foods]

# Limit total number of different foods (encourages focused selection)
min_food_types = 8   # Must select at least 8 different foods
max_food_types = 15  # Cannot select more than 15 different foods

min_foods_constraint = gp.Equation(m, name="min_foods_constraint",
                                  description="Minimum food variety")
min_foods_constraint[:] = gp.Sum(foods, y[foods]) >= min_food_types

max_foods_constraint = gp.Equation(m, name="max_foods_constraint",
                                  description="Maximum food variety")
max_foods_constraint[:] = gp.Sum(foods, y[foods]) <= max_food_types

# Satisfaction constraint (uses actual servings)
satisfaction_constraint = gp.Equation(m, name="satisfaction_constraint",
                                     description="Minimum satisfaction requirement")
satisfaction_constraint[:] = gp.Sum(foods, base_satisfaction[foods] * x[foods]) >= min_satisfaction

In [13]:
# Objective Function (MIP): Minimize cost with selection penalty

# Option 1: Just minimize total cost based on servings
total_cost = gp.Sum(foods, price_per_serving[foods] * x[foods])

# Option 2: Add fixed cost for selecting each food (more realistic)
# selection_cost = 2.0  # Fixed cost for choosing each food type
# obj_expr = total_cost + selection_cost * gp.Sum(foods, y[foods])

# Using Option 1 (pure cost minimization)
obj_expr = total_cost

In [14]:


# Create and solve Mixed-Integer Program (MIP)
diet_plan = gp.Model(
    m,
    equations=m.getEquations(),
    problem=gp.Problem.MIP,  # Changed from LP to MIP
    sense=gp.Sense.MIN,
    objective=obj_expr,
    name="diet_plan_mip",
)

diet_plan.solve(output=sys.stdout)


--- Job _ObLr3w7zQ5KA4N8DgIEtsw.gms Start 12/13/25 01:11:17 52.1.0 4f802a74 WEX-WEI x86 64bit/MS Windows


--- Applying:
    C:\Users\Shashwat\Desktop\CS 524 Introduction to optimization\.venv\Lib\site-packages\gamspy_base\gmsprmNT.txt
--- GAMS Parameters defined
    MIP cplex
    Input C:\Users\Shashwat\AppData\Local\Temp\tmp37duhr26\_ObLr3w7zQ5KA4N8DgIEtsw.gms
    Output C:\Users\Shashwat\AppData\Local\Temp\tmp37duhr26\_ObLr3w7zQ5KA4N8DgIEtsw.lst
    ScrDir C:\Users\Shashwat\AppData\Local\Temp\tmp37duhr26\tmpqzus7hk5\
    SysDir "C:\Users\Shashwat\Desktop\CS 524 Introduction to optimization\.venv\Lib\site-packages\gamspy_base\"
    LogOption 3
    Trace C:\Users\Shashwat\AppData\Local\Temp\tmp37duhr26\_ObLr3w7zQ5KA4N8DgIEtsw.txt
    License C:\Users\Shashwat\Documents\GAMSPy\gamspy_license.txt
    OptFile 0
    OptDir C:\Users\Shashwat\AppData\Local\Temp\tmp37duhr26\
    LimRow 0
    LimCol 0
    TraceOpt 3
    GDX C:\Users\Shashwat\AppData\Local\Temp\tmp37duhr26\_ObLr3w7zQ5KA4N8DgIEtswout.gdx
    SolPrint 0
    SolveLink 2
    PreviousWork 1
    gdxSymbols newOrChanged
System information

,Solver Status,Model Status,Objective,Num of Equations,Num of Variables,Model Type,Solver,Solver Time
0,Normal,OptimalGlobal,18.779888,75,71,MIP,CPLEX,0.031


In [18]:
# Display MIP Results
print("\n" + "="*70)
print("MIXED-INTEGER PROGRAM (MIP) RESULTS")
print("="*70)

# Check if solution is feasible
if diet_plan.status != gp.ModelStatus.OptimalGlobal and diet_plan.status != gp.ModelStatus.OptimalLocal:
    print(f"Status: {diet_plan.status}")
    print("\n⚠️  Model is INFEASIBLE. Possible fixes:")
    print("   1. Run diagnose_mip.py to identify constraint conflicts")
    print("   2. Relax nutrient constraints in nutrient_constraints.csv")
    print("   3. Check VitaminD availability (only few foods have it)")
    print("   4. Lower min_food_types or increase max_food_types in model_config.csv")
    print("   5. See FEASIBILITY_FIXES.md for detailed troubleshooting")
else:
    print(f"Total Cost: ${diet_plan.objective_value:.2f}")
    print(f"Status: {diet_plan.status} ✅")
    
    # Extract solution values from GAMSPy records
    food_col = x.records.columns[0]
    y_col = y.records.columns[0]
    
    x_values = {}
    y_values = {}
    for idx, row in x.records.iterrows():
        x_values[row[food_col]] = row['level']
    for idx, row in y.records.iterrows():
        y_values[row[y_col]] = row['level']
    
    # Get satisfaction and price dictionaries from parameter records
    satisfaction_dict = {}
    for idx, row in base_satisfaction.records.iterrows():
        satisfaction_dict[row['foods']] = row['value']
    
    price_dict = {}
    for idx, row in price_per_serving.records.iterrows():
        price_dict[row['foods']] = row['value']
    
    # Foods selected (binary = 1)
    selected_foods = [food for food, selected in y_values.items() if selected > 0.5]
    total_servings = sum(x_values.values())
    total_satisfaction = sum(satisfaction_dict.get(food, 0) * x_values[food] 
                            for food in selected_foods if x_values[food] > 0)
    
    print("\n" + "="*70)
    print(f"SELECTED FOODS ({len(selected_foods)} items)")
    print("="*70)
    for food in sorted(selected_foods):
        servings = x_values[food]
        if servings > 0:
            cost = price_dict.get(food, 0) * servings
            satisfaction = satisfaction_dict.get(food, 0)
            print(f"{food:35s}: {servings:5.2f} servings | ${cost:6.2f} | sat: {satisfaction:.1f}")
    
    print("\n" + "="*70)
    print("SUMMARY STATISTICS")
    print("="*70)
    print(f"Foods selected: {len(selected_foods)}/{len(foods.toList())}")
    print(f"Total servings: {total_servings:.1f}")
    print(f"Total cost: ${diet_plan.objective_value:.2f}")
    print(f"Total satisfaction: {total_satisfaction:.1f}")
    if total_servings > 0:
        print(f"Avg satisfaction/serving: {total_satisfaction/total_servings:.2f}")
        print(f"Cost per serving: ${diet_plan.objective_value/total_servings:.2f}")


MIXED-INTEGER PROGRAM (MIP) RESULTS
Total Cost: $18.78
Status: ModelStatus.OptimalGlobal ✅

SELECTED FOODS (10 items)
ColdStone_Smoothie                 :  0.52 servings | $  3.11 | sat: 7.0
McDonalds_French_Fries             :  2.44 servings | $  8.52 | sat: 8.5
TacoBell_Crunchwrap                :  0.54 servings | $  2.68 | sat: 8.5
TacoBell_Taco                      :  3.00 servings | $  4.47 | sat: 7.0

SUMMARY STATISTICS
Foods selected: 10/35
Total servings: 6.5
Total cost: $18.78
Total satisfaction: 50.0
Avg satisfaction/serving: 7.69
Cost per serving: $2.89
